In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import MultiPoint
import netCDF4 as nc

# Create a dictionary to replace names of regions

In [2]:
recode_dict = pd.read_csv('Recode dictionary.csv')

In [3]:
# Step 2: Load a world shapefile

directory = 'C:/Users/haley/OneDrive/Graduate/Research/epi_physo/'

world = gpd.read_file(directory + 'naturalearth/ne_110m_admin_0_countries/' + \
                      'ne_110m_admin_0_countries.shp')

In [4]:
recode_dict = recode_dict.drop(['var', 'varnm'], axis = 1)

In [5]:
# String to search for
string_to_find = 'reg'

# Drop rows where 'col1' contains the string
recode_dict = recode_dict[recode_dict['varval'].str.contains(string_to_find, 
                                                             case = False, 
                                                             na = False)]

In [6]:
# Rename countries
recode_dict.replace({'Bolivia (Plurinational State of)' : 'Bolivia'}, 
                    inplace = True)
recode_dict.replace({'China, Hong Kong SAR' : 'China'}, inplace = True)
recode_dict.replace({'China, Macao SAR' : 'China'}, inplace = True)
recode_dict.replace({'China, Taiwan Province of China' : 'Taiwan'}, 
                    inplace = True)
recode_dict.replace({'Congo' : 'Republic of the Congo'},
                    inplace = True)
recode_dict.replace({"Cote d'Ivoire" : "Côte d'Ivoire"}, 
                    inplace = True)
recode_dict.replace({'Czechia' : 'Czech Republic'}, inplace = True)
recode_dict.replace({"Dem. People's Rep. of Korea" : 'Dem. Rep. Korea'}, 
                    inplace = True)
recode_dict.replace({'Dem. Republic of the Congo' : \
                     'Democratic Republic of the Congo'}, 
                    inplace = True)
recode_dict.replace({'Eswatini' : 'Kingdom of eSwatini'}, 
                    inplace = True)
recode_dict.replace({'French Guiana' : 'France'}, inplace = True)
recode_dict.replace({'Gambia' : 'The Gambia'}, inplace = True)
recode_dict.replace({'Iran (Islamic Republic of)' : 'Iran'}, 
                    inplace = True)
recode_dict.replace({"Lao People's Dem. Republic" : 'Lao PDR'}, 
                    inplace = True)
recode_dict.replace({"Republic of Moldova" : 'Moldova'}, 
                    inplace = True)
recode_dict.replace({"State of Palestine" : 'Palestine'}, 
                    inplace = True)
recode_dict.replace({"Syrian Arab Republic" : 'Syria'}, 
                    inplace = True)
recode_dict.replace({'United Republic of Tanzania' : 'Tanzania'}, 
                    inplace = True)
recode_dict.replace({'United States of America' : 'United States'}, 
                    inplace = True)
recode_dict.replace({'Venezuela (Bolivarian Republic of)' : 'Venezuela'}, 
                    inplace = True)
recode_dict.replace({'Viet Nam' : 'Vietnam'}, inplace = True)

In [7]:
list_to_drop = ['Africa', 'Antigua and Barbuda', 'Aruba', 
                'Australia New Zealand', 'Asia', 'Bahrain', 
                'Barbados', 'Cabo Verde', 'Caribbean', 
                'Central America', 'Central Asia', 'Comoros', 
                'Curacao', 'Eastern Africa', 'Eastern Asia', 
                'Eastern Europe', 'Europe', 'French Polynesia', 
                'Grenada', 'Guadeloupe', 'Guam', 'Kiribati', 
                'Latin America and the Caribbean', 'Maldives', 
                'Malta', 'Martinique', 'Mauritius', 'Mayotte', 
                'Melanesia', 'Micronesia', 
                'Micronesia (Fed. States of)', 'Middle Africa', 
                'Northern Africa', 'Northern America', 
                'Northern Europe', 'Oceania', 'Polynesia', 'Reunion', 
                'Saint Lucia', 'Samoa', 'Sao Tome and Principe', 
                'Seychelles', 'Singapore', 'South America', 
                'South-Eastern Asia', 'Southern Africa', 
                'Southern Asia', 'Southern Europe', 
                'St. Vincent and the Grenadines', 'Tonga',
                'United States Virgin Islands', 'Western Africa', 
                'Western Asia', 'Western Europe', 'World']

In [8]:
for country in list_to_drop:
    
    recode_dict = recode_dict[recode_dict['varvaldesc'] != country]

In [9]:
recode_dict = recode_dict.set_index('varval')

In [10]:
recode_dict_dict = recode_dict.to_dict()

In [11]:
recode_dict_final = recode_dict_dict['varvaldesc']

# Load in the actual projections

In [12]:
projections = pd.read_csv(directory + \
                          'iiasa/Feb_17_update_data/PROJresult_AGE_SSP2_V14.csv')

In [13]:
projections = projections.drop(['edu', 'births', 'emi', 'imm', 
                                'deaths'], axis = 1)

In [14]:
# projections = projections.set_index('Time')

In [15]:
projections = projections.replace(recode_dict_final)

In [16]:
# String to search for
string_to_find = 'reg'

# Drop rows where 'col1' contains the string
projections = projections[~projections['region'].str.contains(string_to_find, 
                                                              case = False, 
                                                              na = False)]

In [17]:
projections['agest'] = projections['agest'].astype(str)

In [18]:
projections['Variable'] = projections['sex'] + '_' + \
                          projections['agest']

In [19]:
projections = projections.drop(['sex', 'agest'], axis = 1)

In [20]:
# Reorder columns
projections = projections[['region', 'Variable', 'Time', 'pop']]

In [21]:
# First, let's group the data and sum the population for each unique 
# combination
grouped_df = projections.groupby(['region', 'Variable', 'Time'])['pop'].sum().reset_index()

# Now we can pivot the grouped data
reformatted_df = grouped_df.pivot(index = ['region', 'Variable'], 
                                  columns = 'Time', values = 'pop')

# Reset the index to turn 'region' and 'Variable' back into columns
reformatted_df = reformatted_df.reset_index()

# Back to the old code!

In [22]:
def filter_for_variable(df, variable):

    # Select rows where 'col1' is equal to 'A'
    df_variable = df[df['Variable'] == variable]
    
    df_variable = df_variable.drop(['Variable'], axis = 1)
        
    df_variable = df_variable.set_index('region')
    
    # Add together countries that need to be added together that have 
    # the same name (e.g. France and French Guiana, which was renamed 
    # France)

    df_variable = df_variable.groupby(level = 0).sum()
    
    df_variable.loc['Greenland'] = df_variable.loc['Denmark']
    df_variable.loc['Kosovo'] = df_variable.loc['Serbia']
    df_variable.loc['Northern Cyprus'] = df_variable.loc['Cyprus']
    df_variable.loc['Somaliland'] = df_variable.loc['Somalia']
    
    return df_variable

In [23]:
data_ssp2_Population_Female_neg5_neg1 = filter_for_variable(reformatted_df, 'f_-5')
data_ssp2_Population_Female_00_04 = filter_for_variable(reformatted_df, 'f_0')
data_ssp2_Population_Female_05_09 = filter_for_variable(reformatted_df, 'f_5')
data_ssp2_Population_Female_10_14 = filter_for_variable(reformatted_df, 'f_10')
data_ssp2_Population_Female_15_19 = filter_for_variable(reformatted_df, 'f_15')
data_ssp2_Population_Female_20_24 = filter_for_variable(reformatted_df, 'f_20')
data_ssp2_Population_Female_25_29 = filter_for_variable(reformatted_df, 'f_25')
data_ssp2_Population_Female_30_34 = filter_for_variable(reformatted_df, 'f_30')
data_ssp2_Population_Female_35_39 = filter_for_variable(reformatted_df, 'f_35')
data_ssp2_Population_Female_40_44 = filter_for_variable(reformatted_df, 'f_40')
data_ssp2_Population_Female_45_49 = filter_for_variable(reformatted_df, 'f_45')
data_ssp2_Population_Female_50_54 = filter_for_variable(reformatted_df, 'f_50')
data_ssp2_Population_Female_55_59 = filter_for_variable(reformatted_df, 'f_55')
data_ssp2_Population_Female_60_64 = filter_for_variable(reformatted_df, 'f_60')
data_ssp2_Population_Female_65_69 = filter_for_variable(reformatted_df, 'f_65')
data_ssp2_Population_Female_70_74 = filter_for_variable(reformatted_df, 'f_70')
data_ssp2_Population_Female_75_79 = filter_for_variable(reformatted_df, 'f_75')
data_ssp2_Population_Female_80_84 = filter_for_variable(reformatted_df, 'f_80')
data_ssp2_Population_Female_85_89 = filter_for_variable(reformatted_df, 'f_85')
data_ssp2_Population_Female_90_94 = filter_for_variable(reformatted_df, 'f_90')
data_ssp2_Population_Female_95_99 = filter_for_variable(reformatted_df, 'f_95')
data_ssp2_Population_Female_100_104 = filter_for_variable(reformatted_df, 'f_100')
data_ssp2_Population_Female_105_109 = filter_for_variable(reformatted_df, 'f_105')
data_ssp2_Population_Female_110_114 = filter_for_variable(reformatted_df, 'f_110')
data_ssp2_Population_Female_115_119 = filter_for_variable(reformatted_df, 'f_115')
data_ssp2_Population_Female_120_124 = filter_for_variable(reformatted_df, 'f_120')

In [24]:
data_ssp2_Population_Male_neg5_neg1 = filter_for_variable(reformatted_df, 'm_-5')
data_ssp2_Population_Male_00_04 = filter_for_variable(reformatted_df, 'm_0')
data_ssp2_Population_Male_05_09 = filter_for_variable(reformatted_df, 'm_5')
data_ssp2_Population_Male_10_14 = filter_for_variable(reformatted_df, 'm_10')
data_ssp2_Population_Male_15_19 = filter_for_variable(reformatted_df, 'm_15')
data_ssp2_Population_Male_20_24 = filter_for_variable(reformatted_df, 'm_20')
data_ssp2_Population_Male_25_29 = filter_for_variable(reformatted_df, 'm_25')
data_ssp2_Population_Male_30_34 = filter_for_variable(reformatted_df, 'm_30')
data_ssp2_Population_Male_35_39 = filter_for_variable(reformatted_df, 'm_35')
data_ssp2_Population_Male_40_44 = filter_for_variable(reformatted_df, 'm_40')
data_ssp2_Population_Male_45_49 = filter_for_variable(reformatted_df, 'm_45')
data_ssp2_Population_Male_50_54 = filter_for_variable(reformatted_df, 'm_50')
data_ssp2_Population_Male_55_59 = filter_for_variable(reformatted_df, 'm_55')
data_ssp2_Population_Male_60_64 = filter_for_variable(reformatted_df, 'm_60')
data_ssp2_Population_Male_65_69 = filter_for_variable(reformatted_df, 'm_65')
data_ssp2_Population_Male_70_74 = filter_for_variable(reformatted_df, 'm_70')
data_ssp2_Population_Male_75_79 = filter_for_variable(reformatted_df, 'm_75')
data_ssp2_Population_Male_80_84 = filter_for_variable(reformatted_df, 'm_80')
data_ssp2_Population_Male_85_89 = filter_for_variable(reformatted_df, 'm_85')
data_ssp2_Population_Male_90_94 = filter_for_variable(reformatted_df, 'm_90')
data_ssp2_Population_Male_95_99 = filter_for_variable(reformatted_df, 'm_95')
data_ssp2_Population_Male_100_104 = filter_for_variable(reformatted_df, 'm_100')
data_ssp2_Population_Male_105_109 = filter_for_variable(reformatted_df, 'm_105')
data_ssp2_Population_Male_110_114 = filter_for_variable(reformatted_df, 'm_110')
data_ssp2_Population_Male_115_119 = filter_for_variable(reformatted_df, 'm_115')
data_ssp2_Population_Male_120_124 = filter_for_variable(reformatted_df, 'm_120')

In [25]:
data_ssp2_Population_Female_00_09 = data_ssp2_Population_Female_00_04 + \
                                    data_ssp2_Population_Female_05_09

data_ssp2_Population_Male_00_09 = data_ssp2_Population_Male_00_04 + \
                                  data_ssp2_Population_Male_05_09

In [26]:
data_ssp2_Population_Female_10_64 = data_ssp2_Population_Female_10_14 + \
                                    data_ssp2_Population_Female_15_19 + \
                                    data_ssp2_Population_Female_20_24 + \
                                    data_ssp2_Population_Female_25_29 + \
                                    data_ssp2_Population_Female_30_34 + \
                                    data_ssp2_Population_Female_35_39 + \
                                    data_ssp2_Population_Female_40_44 + \
                                    data_ssp2_Population_Female_45_49 + \
                                    data_ssp2_Population_Female_50_54 + \
                                    data_ssp2_Population_Female_55_59 + \
                                    data_ssp2_Population_Female_60_64

data_ssp2_Population_Male_10_64 = data_ssp2_Population_Male_10_14 + \
                                  data_ssp2_Population_Male_15_19 + \
                                  data_ssp2_Population_Male_20_24 + \
                                  data_ssp2_Population_Male_25_29 + \
                                  data_ssp2_Population_Male_30_34 + \
                                  data_ssp2_Population_Male_35_39 + \
                                  data_ssp2_Population_Male_40_44 + \
                                  data_ssp2_Population_Male_45_49 + \
                                  data_ssp2_Population_Male_50_54 + \
                                  data_ssp2_Population_Male_55_59 + \
                                  data_ssp2_Population_Male_60_64

In [27]:
data_ssp2_Population_Female_65_124 = data_ssp2_Population_Female_65_69 + \
                                     data_ssp2_Population_Female_70_74 + \
                                     data_ssp2_Population_Female_75_79 + \
                                     data_ssp2_Population_Female_80_84 + \
                                     data_ssp2_Population_Female_85_89 + \
                                     data_ssp2_Population_Female_90_94 + \
                                     data_ssp2_Population_Female_95_99 + \
                                     data_ssp2_Population_Female_100_104 + \
                                     data_ssp2_Population_Female_105_109 + \
                                     data_ssp2_Population_Female_110_114 + \
                                     data_ssp2_Population_Female_115_119 + \
                                     data_ssp2_Population_Female_120_124

data_ssp2_Population_Male_65_124 = data_ssp2_Population_Male_65_69 + \
                                   data_ssp2_Population_Male_70_74 + \
                                   data_ssp2_Population_Male_75_79 + \
                                   data_ssp2_Population_Male_80_84 + \
                                   data_ssp2_Population_Male_85_89 + \
                                   data_ssp2_Population_Male_90_94 + \
                                   data_ssp2_Population_Male_95_99 + \
                                   data_ssp2_Population_Male_100_104 + \
                                   data_ssp2_Population_Male_105_109 + \
                                   data_ssp2_Population_Male_110_114 + \
                                   data_ssp2_Population_Male_115_119 + \
                                   data_ssp2_Population_Male_120_124

In [28]:
data_ssp2_Population_Female_65_84 = data_ssp2_Population_Female_65_69 + \
                                    data_ssp2_Population_Female_70_74 + \
                                    data_ssp2_Population_Female_75_79 + \
                                    data_ssp2_Population_Female_80_84 

data_ssp2_Population_Male_65_84 = data_ssp2_Population_Male_65_69 + \
                                  data_ssp2_Population_Male_70_74 + \
                                  data_ssp2_Population_Male_75_79 + \
                                  data_ssp2_Population_Male_80_84

In [29]:
data_ssp2_Population_Female_85_124 = data_ssp2_Population_Female_85_89 + \
                                     data_ssp2_Population_Female_90_94 + \
                                     data_ssp2_Population_Female_95_99 + \
                                     data_ssp2_Population_Female_100_104 + \
                                     data_ssp2_Population_Female_105_109 + \
                                     data_ssp2_Population_Female_110_114 + \
                                     data_ssp2_Population_Female_115_119 + \
                                     data_ssp2_Population_Female_120_124

data_ssp2_Population_Male_85_124 = data_ssp2_Population_Male_85_89 + \
                                   data_ssp2_Population_Male_90_94 + \
                                   data_ssp2_Population_Male_95_99 + \
                                   data_ssp2_Population_Male_100_104 + \
                                   data_ssp2_Population_Male_105_109 + \
                                   data_ssp2_Population_Male_110_114 + \
                                   data_ssp2_Population_Male_115_119 + \
                                   data_ssp2_Population_Male_120_124

In [30]:
data_ssp2_Population_Female = data_ssp2_Population_Female_00_09 + \
                              data_ssp2_Population_Female_10_64 + \
                              data_ssp2_Population_Female_65_124

data_ssp2_Population_Male = data_ssp2_Population_Male_00_09 + \
                            data_ssp2_Population_Male_10_64 + \
                            data_ssp2_Population_Male_65_124

In [31]:
data_ssp2_Population = data_ssp2_Population_Female + \
                       data_ssp2_Population_Male

In [32]:
data_children_ratio = (data_ssp2_Population_Female_00_09 + \
                       data_ssp2_Population_Male_00_09) / \
                       data_ssp2_Population
data_young_ratio = (data_ssp2_Population_Female_10_64 + \
                    data_ssp2_Population_Male_10_64) / \
                    data_ssp2_Population
data_old_ratio = (data_ssp2_Population_Female_65_124 + \
                  data_ssp2_Population_Male_65_124) / \
                  data_ssp2_Population

In [33]:
data_children_ratio

Time,2020,2025,2030,2035,2040,2045,2050,2055,2060,2065,2070,2075,2080,2085,2090,2095,2100
region,,,,,,,,,,,,,,,,,
Afghanistan,0.310026,0.297019,0.281223,0.264657,0.247159,0.230686,0.215558,0.201301,0.187920,0.175770,0.164082,0.152829,0.141810,0.130850,0.122816,0.118191,0.114497
Albania,0.107949,0.111089,0.111400,0.097972,0.087981,0.082902,0.081395,0.080848,0.079213,0.076225,0.073167,0.071381,0.070921,0.071024,0.071016,0.070610,0.070117
Algeria,0.219648,0.205173,0.179110,0.157940,0.147515,0.144090,0.141408,0.133986,0.122201,0.110575,0.102498,0.099388,0.098464,0.096696,0.092787,0.087707,0.083449
Angola,0.327087,0.312746,0.292858,0.274598,0.259332,0.243269,0.226608,0.210728,0.196547,0.183647,0.170473,0.158337,0.146994,0.135601,0.124829,0.117167,0.113078
Argentina,0.159419,0.140721,0.125034,0.120157,0.116280,0.111768,0.106697,0.101977,0.096950,0.092362,0.089657,0.087998,0.086340,0.084391,0.082401,0.080507,0.079027
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Zimbabwe,0.290395,0.265363,0.243762,0.226931,0.212057,0.197887,0.183887,0.169686,0.156404,0.143829,0.132732,0.125194,0.120853,0.117072,0.112645,0.108209,0.104642
Greenland,0.105045,0.104538,0.105829,0.107551,0.106719,0.105387,0.105732,0.106091,0.105457,0.105349,0.104681,0.103378,0.102021,0.100862,0.099764,0.098615,0.097328
Kosovo,0.095510,0.096274,0.093258,0.090755,0.088365,0.086975,0.087442,0.088010,0.087517,0.086189,0.084821,0.083784,0.083170,0.082689,0.081831,0.080401,0.078663


# Grid the population data

In [34]:
# Step 2: Load a world shapefile
world = gpd.read_file(directory + 'naturalearth/ne_110m_admin_0_countries/' + \
                      'ne_110m_admin_0_countries.shp')

In [35]:
def create_data(data, grid_res, year):
    
    new_world = world.merge(data, how = 'left', 
                            left_on = ['NAME_LONG'], 
                            right_on = ['region'])
    
    # Step 4: Create a grid of lat/lon points
    lat = np.arange(-90, 90, grid_res)  # 0.5 degree resolution, 
                                        # adjust as needed
    lon = np.arange(-180, 180, grid_res)
    
    # Create a MultiPoint object from the grid
    points = MultiPoint([(x, y) for x in lon for y in lat])

    # Step 5: Create an empty array to store population data
    data = np.full((len(lat), len(lon)), np.nan)
    
    # Step 6: Fill the population array
    for idx, country in new_world.iterrows():
        if pd.notna(country[year]):  # Check if population data exists
            mask = country.geometry.intersects(points)
            if mask:
                country_points = points.intersection(country.geometry)
                for point in country_points.geoms:
                    i = np.argmin(np.abs(lon - point.x))
                    j = np.argmin(np.abs(lat - point.y))
                    data[j, i] = country[year]
                    
    return data, lat, lon

In [36]:
%%time

data = data_children_ratio 
grid_res = 0.25
year = 2020
data, lat, lon = create_data(data, grid_res, year)
name = 'ratio_children_' + str(year) + '_grid.nc'
description = str(year) + ' ratio of people aged 00-09 : total people'
var_units = 'unitless'

CPU times: total: 21min 2s
Wall time: 21min 45s


In [37]:
with nc.Dataset(name, 'w', format = 'NETCDF4') as nc_out:
    # Define dimensions
    nc_out.createDimension('lat', len(lat))
    nc_out.createDimension('lon', len(lon))

    # Create variables
    latitudes = nc_out.createVariable('lat', 'f4', ('lat',))
    longitudes = nc_out.createVariable('lon', 'f4', ('lon',))
    var = nc_out.createVariable(str(year), 'f4', ('lat', 'lon',))

    # Add data
    latitudes[:] = lat
    longitudes[:] = lon
    var[:] = data

    # Add attributes
    nc_out.description = description
    latitudes.units = 'degrees_north'
    longitudes.units = 'degrees_east'
    var.units = var_units

In [38]:
%%time

data = data_young_ratio 
grid_res = 0.25
year = 2020
data, lat, lon = create_data(data, grid_res, year)
name = 'ratio_young_' + str(year) + '_grid.nc'
description = str(year) + ' ratio of people aged 10-64 : total people'
var_units = 'unitless'

CPU times: total: 14min 44s
Wall time: 15min 9s


In [39]:
with nc.Dataset(name, 'w', format = 'NETCDF4') as nc_out:
    # Define dimensions
    nc_out.createDimension('lat', len(lat))
    nc_out.createDimension('lon', len(lon))

    # Create variables
    latitudes = nc_out.createVariable('lat', 'f4', ('lat',))
    longitudes = nc_out.createVariable('lon', 'f4', ('lon',))
    var = nc_out.createVariable(str(year), 'f4', ('lat', 'lon',))

    # Add data
    latitudes[:] = lat
    longitudes[:] = lon
    var[:] = data

    # Add attributes
    nc_out.description = description
    latitudes.units = 'degrees_north'
    longitudes.units = 'degrees_east'
    var.units = var_units

In [40]:
%%time

data = data_old_ratio 
grid_res = 0.25
year = 2020
data, lat, lon = create_data(data, grid_res, year)
name = 'ratio_old_' + str(year) + '_grid.nc'
description = str(year) + ' ratio of people aged 65-124 : total people'
var_units = 'unitless'

CPU times: total: 7min 19s
Wall time: 7min 24s


In [41]:
with nc.Dataset(name, 'w', format = 'NETCDF4') as nc_out:
    # Define dimensions
    nc_out.createDimension('lat', len(lat))
    nc_out.createDimension('lon', len(lon))

    # Create variables
    latitudes = nc_out.createVariable('lat', 'f4', ('lat',))
    longitudes = nc_out.createVariable('lon', 'f4', ('lon',))
    var = nc_out.createVariable(str(year), 'f4', ('lat', 'lon',))

    # Add data
    latitudes[:] = lat
    longitudes[:] = lon
    var[:] = data

    # Add attributes
    nc_out.description = description
    latitudes.units = 'degrees_north'
    longitudes.units = 'degrees_east'
    var.units = var_units